<div style="font-size: 13px; line-height: 1.25;">

# CEPII Gravity — One-Time Pre-Processing

This notebook slims the full CEPII Gravity v202211 file (`Gravity_V202211.csv`, ~1.22 GB) down to a thesis-ready subset (~30-50 MB) by selecting only the columns we need and the years 2000-2019.

**Run this notebook once.** The output file `cepii_gravity_subset.csv` is what the main gravity notebook (`dti_gravity_ppml.ipynb`) reads. You do not need to re-run this unless you change the variable selection or year range.

### What this does

1. Reads the full CEPII Gravity CSV (1.22 GB, ~4.6 million rows × ~80 columns).
2. Selects 22 columns: geography (`dist`, `contig`, `comlang_off`, `colony`, `col_dep_ever`, `comrelig`), institutional membership (`gatt_o/d`, `wto_o/d`, `eu_o/d`), trade agreements (`fta_wto`, `rta_coverage`, `rta_type`), social connectedness (`scaled_sci_2021`), and identifier keys (`year`, `iso3_o`, `iso3_d`, `country_exists_o/d`).
3. Filters to 2000-2019 (the panel range your gravity notebook uses) and to rows where both countries exist (drops null pairs from territorial-reconfiguration columns).
4. Writes a slim CSV to the location the main notebook expects.

### Requirements

* R kernel
* Packages `readr` and `dplyr` (both standard tidyverse — installed automatically if missing).
* The full CEPII Gravity file extracted to the path defined in `INPUT_CSV` below.

</div>

<div style="font-size: 25px; line-height: 1.25;">
SECTION 0 — SETUP
</div>

Load packages and define paths. Adjust `INPUT_CSV` if your CEPII zip was extracted to a different location.

In [ ]:
# --- Required packages --------------------------------------------------------

required_packages <- c("readr", "dplyr")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

cat("Packages loaded.\n")

# --- Configuration block ------------------------------------------------------

# Input: the full CEPII Gravity CSV extracted from the downloaded zip.
INPUT_CSV  <- "../data/raw/Gravity_csv_V202211/Gravity_V202211.csv"

# Output: the slim subset that the main gravity notebook will read.
OUTPUT_CSV <- "../data/final_dataset/cepii_gravity_subset.csv"

# Year range — keep this aligned with your Comtrade panel; CEPII Gravity ends 2019.
YEAR_MIN <- 2000
YEAR_MAX <- 2019

# Columns to keep from the full file.
keep_cols <- c(
  # Keys
  "year", "iso3_o", "iso3_d", "country_exists_o", "country_exists_d",
  # Geographic / cultural (time-invariant within pair)
  "dist", "contig", "comlang_off", "col45", "col_dep_ever", "comrelig",
  # Trade-facilitation / membership (time-varying — source for `both_*` indicators)
  "gatt_o", "gatt_d", "wto_o", "wto_d", "eu_o", "eu_d",
  "fta_wto", "rta_coverage", "rta_type",
  # Social connectedness — single 2021 cross-section, time-invariant by construction
  "scaled_sci_2021"
)

cat(sprintf("\n  Input file:  %s\n", INPUT_CSV))
cat(sprintf("  Output file: %s\n", OUTPUT_CSV))
cat(sprintf("  Year range:  %d-%d\n", YEAR_MIN, YEAR_MAX))
cat(sprintf("  Columns:     %d kept\n", length(keep_cols)))

# Sanity check input exists before we try to read 1.2 GB.
if (!file.exists(INPUT_CSV)) {
  stop(sprintf("Input CSV not found at:\n  %s\nUpdate INPUT_CSV above.", INPUT_CSV))
}
cat(sprintf("\n  Input file size: %.2f GB\n", file.info(INPUT_CSV)$size / 1e9))

<div style="font-size: 25px; line-height: 1.25;">
SECTION 1 — READ THE FULL GRAVITY CSV
</div>

`readr::read_csv` with `col_select` is memory-efficient: it loads only the requested columns rather than the full ~80-column file. Expected wall time: 2-4 minutes depending on your machine.

In [ ]:
# --- 1.1 Load the requested columns from the full file ----------------------

cat("Reading CEPII Gravity CSV (~1.2 GB) — this takes ~2-4 minutes.\n")
t0 <- Sys.time()

gravity <- read_csv(
  INPUT_CSV,
  col_select     = all_of(keep_cols),
  col_types      = cols(.default = col_guess()),
  show_col_types = FALSE,
  progress       = TRUE
)

elapsed <- as.numeric(Sys.time() - t0, units = "secs")
cat(sprintf("\nLoaded %s rows x %d columns in %.1f seconds (%.1f minutes)\n",
            format(nrow(gravity), big.mark = ","),
            ncol(gravity),
            elapsed, elapsed / 60))

<div style="font-size: 25px; line-height: 1.25;">
SECTION 2 — FILTER AND WRITE THE SUBSET
</div>

Restrict to (i) years in the target range and (ii) rows where BOTH countries exist in that year (CEPII is "squared" — every country pair × year combination appears even when one country didn't yet exist; the `country_exists_*` flags identify these).

After filtering, drop the `country_exists_*` helper columns since they're constant within the subset (always 1).


In [ ]:
# --- 2.1 Apply filters --------------------------------------------------------

n_raw <- nrow(gravity)

gravity_sub <- gravity %>%
  filter(year >= YEAR_MIN, year <= YEAR_MAX,
         country_exists_o == 1, country_exists_d == 1) %>%
  select(-country_exists_o, -country_exists_d)

cat(sprintf("Filter result:\n"))
cat(sprintf("  Before: %s rows\n", format(n_raw,             big.mark = ",")))
cat(sprintf("  After:  %s rows  (%.1f%% retained)\n",
            format(nrow(gravity_sub), big.mark = ","),
            100 * nrow(gravity_sub) / n_raw))
cat(sprintf("  Year coverage:  %d to %d\n",
            min(gravity_sub$year, na.rm = TRUE),
            max(gravity_sub$year, na.rm = TRUE)))
cat(sprintf("  Unique origins: %d\n", length(unique(gravity_sub$iso3_o))))
cat(sprintf("  Unique destinations: %d\n", length(unique(gravity_sub$iso3_d))))

In [ ]:
# --- 2.2 Write the slim CSV --------------------------------------------------

# Make sure the output directory exists.
out_dir <- dirname(OUTPUT_CSV)
if (!dir.exists(out_dir)) {
  cat(sprintf("Creating output directory: %s\n", out_dir))
  dir.create(out_dir, recursive = TRUE)
}

write_csv(gravity_sub, OUTPUT_CSV)

cat(sprintf("\nSaved: %s\n", OUTPUT_CSV))
cat(sprintf("Output size: %.1f MB\n", file.info(OUTPUT_CSV)$size / 1e6))
cat(sprintf("Compression vs input: %.1fx smaller\n",
            file.info(INPUT_CSV)$size / file.info(OUTPUT_CSV)$size))

<div style="font-size: 25px; line-height: 1.25;">
SECTION 3 — SANITY CHECK
</div>

Quick post-write inspection: re-read the subset and print head + variable summaries to confirm the file is healthy.

In [ ]:
# --- 3.1 Re-read the subset and inspect --------------------------------------

verify <- read_csv(OUTPUT_CSV, show_col_types = FALSE, n_max = 5)
cat("First 5 rows of the subset:\n")
print(verify)

cat("\n--- Column types ---\n")
str(verify, give.attr = FALSE)

cat("\n--- Sample non-NA rates (first 10,000 rows of subset) ---\n")
verify_full <- read_csv(OUTPUT_CSV, show_col_types = FALSE, n_max = 10000)
non_na_rates <- sapply(verify_full, function(x) mean(!is.na(x)))
print(round(non_na_rates * 100, 1))

cat("\nDone. The slim subset is ready to be read by `dti_gravity_ppml.ipynb`.\n")
cat("You do not need to re-run this notebook unless you change `keep_cols`,\n")
cat("`YEAR_MIN`, or `YEAR_MAX` in Section 0.\n")